# Evidências sobre o desempenho da EWMA3

Este notebook investiga por que os modelos têm dificuldade para melhorar a EWMA3. Ele reúne evidências fora da amostra sobre: desempenho por alvo e fold; persistência mês a mês dentro de cada H3 (removendo o nível médio da célula); previsão dos erros da EWMA3; efeito da divisão alto/baixo; concentração de ocorrências no grupo alto; e importância das features para o resíduo.

Os folds são jan–jun/2024, jul–dez/2024 e jan–jun/2025. 2022 pode fornecer histórico anterior, mas não é período de treino. Jul/2025–jun/2026 segue reservado. A validação existente é rolling one-step-ahead: observações mensais anteriores dentro do período de validação alimentam o histórico para o mês seguinte.

O notebook separa **evidência** (medidas calculadas nos dados) de **interpretação** (hipóteses sobre por que ocorre). Correlação e importância preditiva não demonstram causalidade. Com três folds, diferenças pequenas devem ser tratadas como frágeis.

Execute a célula para gerar tabelas e CSVs com prefixo `evidence_why_ewm3_h3_` em `datasets/`.

## Evidências já calculadas

| Alvo | RMSE média histórica fixa | RMSE EWMA3 | Redução de RMSE da EWMA3 | Correlação intra-H3: atual vs lag 1 | Correlação intra-H3: atual vs EWMA3 |
|---|---:|---:|---:|---:|---:|
| CRIME | 0,02234 | 0,02036 | 8,82% | 0,33 | 0,40 |
| FURTO | 0,02422 | 0,02327 | 3,95% | 0,24 | 0,29 |
| ROUBO | 0,03533 | 0,03038 | 13,74% | 0,43 | 0,48 |

O grupo alto (20% dos H3) concentrou em média **90% de CRIME, 92% de FURTO e 89% de ROUBO** no treino. Apesar dessa concentração, separar os grupos não garantiu que os algoritmos superassem a EWMA3.

No Ridge residual agrupado, a correlação entre o erro real da EWMA3 e o ajuste previsto foi apenas **0,15 para CRIME, 0,15 para FURTO e 0,18 para ROUBO**. O ganho médio sobre a EWMA3 foi **0,68%, 0,02% e 2,38%**, respectivamente; esses ganhos ocorreram em 2/3, 1/3 e 3/3 folds. Isso aponta que ainda há pouco sinal estável para corrigir a previsão recente, principalmente em FURTO.

In [1]:
from pathlib import Path
import runpy
script = Path('processing_datasets/evidence_why_ewm3_strong_h3.py')
if not script.exists():
    script = Path('evidence_why_ewm3_strong_h3.py')
runpy.run_path(str(script), run_name='__main__')

1) Baselines e melhores algoritmos por escopo:
     target                   escopo                             modelo  RMSE_medio  ganho_medio_vs_EWM3_pct  folds_melhor_EWM3  ganho_medio_vs_media_fixa_pct
RISCO_CRIME alto_modelo_baixo_hurdle    Ridge_log1p_alto_HurdleNB_baixo    0.020959                -3.026900                  1                       6.041862
RISCO_CRIME                 baseline                               EWM3    0.020363                 0.000000                  0                       8.824851
RISCO_CRIME                 baseline               media_historica_fixa    0.022339                -9.717902                  0                       0.000000
RISCO_CRIME                   global                          RF_global    0.021301                -4.732520                  1                       4.476717
RISCO_CRIME      grupos_mesmo_modelo    Ridge_log1p_grupos_mesmo_modelo    0.021112                -3.779963                  1                       5.354312

{'__name__': '__main__',
 '__doc__': 'Create empirical evidence tables for why EWMA3 is a strong benchmark.',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'evidence_why_ewm3_strong_h3.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __impor

## Como interpretar as evidências

A EWM3 vence a média histórica fixa nos três alvos. Isso demonstra que atualizar o nível de cada célula com observações recentes é valioso. A correlação temporal relevante é a calculada após subtrair o nível médio de treino de cada H3; correlações brutas misturam persistência com diferenças permanentes entre células.

O grupo alto concentra grande parte das ocorrências, mas isso não garante que seja previsível além da EWM3. Para demonstrar ganho, o modelo precisa prever a parte que a EWM3 erra. As tabelas de resíduos medem essa correção fora da amostra; se a correção acompanha pouco os erros reais ou não reduz RMSE repetidamente, as features atuais não acrescentam sinal estável suficiente.

Esses resultados sustentam uma explicação provável, não provam que nenhum modelo futuro poderá superar a EWM3. Há só três folds; a avaliação também é rolling de um mês à frente. Para um horizonte de vários meses sem atualização dos registros, será necessária uma validação de origem fixa.